# 08 — Demand Prediction

Mục tiêu: dự đoán **số xe đang có trong bãi** tại một thời điểm.

Feature dùng cho model:
- `hour`
- `day_of_week`
- `is_weekend`
- `is_morning`
- `is_exam_week`

Dataset được tổng hợp theo từng giờ từ thời gian vào/ra xe.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

raw_path = '../data/raw/parking_data_raw.csv'
df = pd.read_csv(raw_path)

df['entry_time'] = pd.to_datetime(df['entry_time'])
df['exit_time'] = pd.to_datetime(df['exit_time'])

print("Dữ liệu raw:", df.shape)
display(df.head())

In [ ]:
start_hour = df['entry_time'].dt.floor('h').min()
end_hour = df['exit_time'].dt.floor('h').max()

hourly_index = pd.date_range(
    start=start_hour,
    end=end_hour,
    freq='h'
)

entries = (
    df['entry_time']
    .dt.floor('h')
    .value_counts()
    .sort_index()
)

exits = (
    df['exit_time']
    .dt.floor('h')
    .value_counts()
    .sort_index()
)

events = (
    entries.reindex(hourly_index, fill_value=0)
    - exits.reindex(hourly_index, fill_value=0)
)

occupancy = events.cumsum().clip(lower=0)

demand_df = pd.DataFrame({
    'timestamp': hourly_index,
    'demand': occupancy.astype(int).to_numpy()
})

demand_df['hour'] = demand_df['timestamp'].dt.hour
demand_df['day_of_week'] = demand_df['timestamp'].dt.weekday
demand_df['is_weekend'] = (
    demand_df['day_of_week'] >= 5
).astype(int)
demand_df['is_morning'] = (
    demand_df['hour'] < 12
).astype(int)
demand_df['is_exam_week'] = (
    (demand_df['timestamp'].dt.month == 12)
    & (demand_df['timestamp'].dt.day > 10)
).astype(int)

print("Kích thước demand dataset:", demand_df.shape)
display(demand_df.head(10))

In [ ]:
feature_columns = [
    'hour',
    'day_of_week',
    'is_weekend',
    'is_morning',
    'is_exam_week'
]

X = demand_df[feature_columns]
y = demand_df['demand']

print("Features:")
print(feature_columns)

print("\nTarget: demand")
print(y.describe().round(2))

display(demand_df[feature_columns + ['demand']].head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"Đang huấn luyện: {name}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        'model': name,
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    })

    trained_models[name] = model

    print(f"MAE : {mae:.2f} xe")
    print(f"RMSE: {rmse:.2f} xe")
    print(f"R²  : {r2:.4f}")
    print("-" * 50)

results_df = (
    pd.DataFrame(results)
    .sort_values(
        by=['mae', 'rmse'],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("\nBảng so sánh Demand Prediction:")
display(results_df.round(4))

In [ ]:
best_model_name = results_df.loc[0, 'model']
best_demand_model = trained_models[best_model_name]

y_pred_best = best_demand_model.predict(X_test)

print("Model tốt nhất:", best_model_name)
print("MAE :", round(mean_absolute_error(y_test, y_pred_best), 2), "xe")
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_best)), 2), "xe")
print("R²  :", round(r2_score(y_test, y_pred_best), 4))

comparison = pd.DataFrame({
    'actual_demand': y_test.to_numpy(),
    'predicted_demand': np.round(y_pred_best, 2)
})

display(comparison.head(10))

In [ ]:
os.makedirs('../models/demand', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

demand_data_path = '../data/processed/demand_data.csv'
model_path = '../models/demand/best_demand_model.pkl'
results_path = '../data/processed/demand_model_results.csv'

demand_df.to_csv(demand_data_path, index=False)
joblib.dump(best_demand_model, model_path)
results_df.to_csv(results_path, index=False)

print("Đã lưu demand dataset:")
print(demand_data_path)

print("\nĐã lưu model tốt nhất:")
print(model_path)

print("\nĐã lưu kết quả:")
print(results_path)

print("\nThứ tự feature của model:")
print(best_demand_model.feature_names_in_.tolist())

In [ ]:
sample = X_test.iloc[[0]]
prediction = max(
    0,
    int(round(best_demand_model.predict(sample)[0]))
)

print("Input:")
display(sample)

print("Số xe dự đoán đang trong bãi:", prediction)